# Pipeline de entrenamiento CNN para MNIST

Este notebook transforma el ejercicio original en un **pipeline reutilizable de entrenamiento**.

La idea es que el flujo quede ordenado y repetible:

1. Configuración centralizada.
2. Carga y preparación de datos.
3. Construcción del modelo.
4. Compilación.
5. Entrenamiento con callbacks.
6. Evaluación.
7. Guardado de artefactos del experimento.

**Artefactos generados:**

- `best_model.keras`
- `final_model.keras`
- `training_history.csv`
- `metrics.json`
- `classification_report.txt`
- `loss_curve.png`
- `accuracy_curve.png`
- `confusion_matrix.png`


## 0. Instalación opcional

Ejecutar solo si el entorno no tiene las librerías instaladas.

In [ ]:
# !pip install tensorflow scikit-learn pandas matplotlib

## 1. Importar librerías

In [ ]:
from __future__ import annotations

import json
import os
import random
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)

## 2. Configuración central del pipeline

Cambiar estos valores permite repetir el experimento sin modificar todo el código.

In [ ]:
@dataclass
class TrainingConfig:
    model_name: str = "mnist_cnn"
    artifacts_dir: str = "artifacts_mnist_cnn"
    random_state: int = 42
    validation_size: float = 0.10

    image_height: int = 28
    image_width: int = 28
    channels: int = 1
    num_classes: int = 10

    learning_rate: float = 0.001
    batch_size: int = 128
    epochs: int = 10
    patience: int = 3
    monitor_metric: str = "val_loss"


config = TrainingConfig()
config

## 3. Funciones auxiliares

In [ ]:
def set_seed(seed: int) -> None:
    """Fija semillas para favorecer reproducibilidad."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def create_artifacts_dir(config: TrainingConfig) -> Path:
    """Crea la carpeta de resultados del experimento."""
    artifacts_path = Path(config.artifacts_dir)
    artifacts_path.mkdir(parents=True, exist_ok=True)
    return artifacts_path


def save_config(config: TrainingConfig, artifacts_path: Path) -> None:
    """Guarda la configuración usada en el experimento."""
    with open(artifacts_path / "config.json", "w", encoding="utf-8") as f:
        json.dump(asdict(config), f, indent=4, ensure_ascii=False)

## 4. Carga y preparación del dataset

In [ ]:
def load_and_prepare_data(
    config: TrainingConfig,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Carga MNIST, normaliza, agrega canal y separa validación."""
    (x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

    print("Shape original x_train:", x_train.shape)
    print("Shape original x_test :", x_test.shape)

    # Normalización: de 0-255 a 0-1
    x_train = x_train.astype("float32") / 255.0
    x_test = x_test.astype("float32") / 255.0

    # Agregar canal: (n, 28, 28) -> (n, 28, 28, 1)
    x_train = np.expand_dims(x_train, axis=-1)
    x_test = np.expand_dims(x_test, axis=-1)

    x_train, x_val, y_train, y_val = train_test_split(
        x_train,
        y_train,
        test_size=config.validation_size,
        random_state=config.random_state,
        stratify=y_train,
    )

    print("Shape final x_train:", x_train.shape)
    print("Shape final x_val  :", x_val.shape)
    print("Shape final x_test :", x_test.shape)

    return x_train, x_val, x_test, y_train, y_val, y_test

In [ ]:
set_seed(config.random_state)
artifacts_path = create_artifacts_dir(config)
save_config(config, artifacts_path)

x_train, x_val, x_test, y_train, y_val, y_test = load_and_prepare_data(config)

## 5. Visualización rápida de datos

In [ ]:
num_images = 8
plt.figure(figsize=(12, 3))

for i in range(num_images):
    plt.subplot(1, num_images, i + 1)
    plt.imshow(x_train[i].squeeze(), cmap="gray")
    plt.title(f"y={y_train[i]}")
    plt.axis("off")

plt.suptitle("Ejemplos del dataset MNIST")
plt.tight_layout()
plt.show()

## 6. Construcción y compilación del modelo

In [ ]:
def build_model(config: TrainingConfig) -> keras.Model:
    """Construye una CNN para clasificación de dígitos MNIST."""
    input_shape = (config.image_height, config.image_width, config.channels)

    model = keras.Sequential(
        [
            layers.Input(shape=input_shape),

            layers.Conv2D(32, kernel_size=3, activation="relu", padding="same"),
            layers.MaxPooling2D(pool_size=2),

            layers.Conv2D(64, kernel_size=3, activation="relu", padding="same"),
            layers.MaxPooling2D(pool_size=2),

            layers.Flatten(),
            layers.Dense(64, activation="relu"),
            layers.Dropout(0.25),

            layers.Dense(config.num_classes, activation="softmax"),
        ],
        name=config.model_name,
    )

    return model


def compile_model(model: keras.Model, config: TrainingConfig) -> keras.Model:
    """Compila el modelo."""
    optimizer = keras.optimizers.Adam(learning_rate=config.learning_rate)

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model

In [ ]:
model = build_model(config)
model = compile_model(model, config)
model.summary()

## 7. Callbacks

In [ ]:
def get_callbacks(config: TrainingConfig, artifacts_path: Path) -> list:
    """Define callbacks para detener, guardar y ajustar learning rate."""
    checkpoint_path = artifacts_path / "best_model.keras"

    early_stopping = keras.callbacks.EarlyStopping(
        monitor=config.monitor_metric,
        patience=config.patience,
        restore_best_weights=True,
        verbose=1,
    )

    checkpoint = keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor=config.monitor_metric,
        save_best_only=True,
        verbose=1,
    )

    reduce_lr = keras.callbacks.ReduceLROnPlateau(
        monitor=config.monitor_metric,
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    )

    return [early_stopping, checkpoint, reduce_lr]


callbacks = get_callbacks(config, artifacts_path)
callbacks

## 8. Entrenamiento

In [ ]:
history = model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=config.epochs,
    batch_size=config.batch_size,
    callbacks=callbacks,
    verbose=2,
)

history_df = pd.DataFrame(history.history)
history_df.to_csv(artifacts_path / "training_history.csv", index=False)
history_df

## 9. Diagnóstico del entrenamiento

In [ ]:
def plot_training_curves(history: keras.callbacks.History, artifacts_path: Path) -> None:
    """Grafica y guarda curvas de entrenamiento."""
    history_df = pd.DataFrame(history.history)
    epochs_range = range(1, len(history_df) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs_range, history_df["loss"], label="Pérdida entrenamiento")
    plt.plot(epochs_range, history_df["val_loss"], label="Pérdida validación")
    plt.xlabel("Épocas")
    plt.ylabel("Pérdida")
    plt.title("Curva de pérdida")
    plt.legend()
    plt.tight_layout()
    plt.savefig(artifacts_path / "loss_curve.png", dpi=150)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs_range, history_df["accuracy"], label="Accuracy entrenamiento")
    plt.plot(epochs_range, history_df["val_accuracy"], label="Accuracy validación")
    plt.xlabel("Épocas")
    plt.ylabel("Accuracy")
    plt.title("Curva de accuracy")
    plt.legend()
    plt.tight_layout()
    plt.savefig(artifacts_path / "accuracy_curve.png", dpi=150)
    plt.show()


plot_training_curves(history, artifacts_path)

## 10. Evaluación del modelo

In [ ]:
def evaluate_model(
    model: keras.Model,
    x_test: np.ndarray,
    y_test: np.ndarray,
    artifacts_path: Path,
) -> dict:
    """Evalúa el modelo y guarda métricas/reporte/matriz de confusión."""
    test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)

    y_pred_probs = model.predict(x_test, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)

    report_dict = classification_report(y_test, y_pred, output_dict=True)
    report_text = classification_report(y_test, y_pred)

    metrics = {
        "test_loss": float(test_loss),
        "test_accuracy": float(test_accuracy),
        "macro_f1": float(report_dict["macro avg"]["f1-score"]),
        "weighted_f1": float(report_dict["weighted avg"]["f1-score"]),
    }

    with open(artifacts_path / "metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=4, ensure_ascii=False)

    with open(artifacts_path / "classification_report.txt", "w", encoding="utf-8") as f:
        f.write(report_text)

    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(8, 6))
    plt.imshow(cm)
    plt.title("Matriz de confusión - MNIST")
    plt.xlabel("Etiqueta predicha")
    plt.ylabel("Etiqueta real")
    plt.colorbar()

    tick_marks = np.arange(10)
    plt.xticks(tick_marks, tick_marks)
    plt.yticks(tick_marks, tick_marks)

    threshold = cm.max() / 2
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(
                j,
                i,
                format(cm[i, j], "d"),
                ha="center",
                va="center",
                color="white" if cm[i, j] > threshold else "black",
                fontsize=8,
            )

    plt.tight_layout()
    plt.savefig(artifacts_path / "confusion_matrix.png", dpi=150)
    plt.show()

    return metrics, report_text


metrics, report_text = evaluate_model(model, x_test, y_test, artifacts_path)

print("Métricas finales:")
print(json.dumps(metrics, indent=4, ensure_ascii=False))
print("\nReporte de clasificación:")
print(report_text)

## 11. Guardado del modelo final

In [ ]:
final_model_path = artifacts_path / "final_model.keras"
model.save(final_model_path)

print(f"Modelo final guardado en: {final_model_path}")
print(f"Mejor modelo guardado en: {artifacts_path / 'best_model.keras'}")
print(f"Artefactos guardados en: {artifacts_path.resolve()}")

## 12. Función única para ejecutar todo el pipeline

Esta función permite reutilizar el flujo completo con otra configuración.

In [ ]:
def run_training_pipeline(config: TrainingConfig) -> dict:
    """Ejecuta el pipeline completo de entrenamiento."""
    set_seed(config.random_state)

    artifacts_path = create_artifacts_dir(config)
    save_config(config, artifacts_path)

    x_train, x_val, x_test, y_train, y_val, y_test = load_and_prepare_data(config)

    model = build_model(config)
    model = compile_model(model, config)

    history = model.fit(
        x_train,
        y_train,
        validation_data=(x_val, y_val),
        epochs=config.epochs,
        batch_size=config.batch_size,
        callbacks=get_callbacks(config, artifacts_path),
        verbose=2,
    )

    pd.DataFrame(history.history).to_csv(
        artifacts_path / "training_history.csv",
        index=False,
    )

    plot_training_curves(history, artifacts_path)
    metrics, _ = evaluate_model(model, x_test, y_test, artifacts_path)

    model.save(artifacts_path / "final_model.keras")

    return metrics


# Ejemplo de uso:
# nueva_config = TrainingConfig(epochs=5, batch_size=64, learning_rate=0.0005)
# run_training_pipeline(nueva_config)